# Congress House Bills Web Scraper

## Description

A web scraper for Congress House bills (HBNXXXXX)

## Imports and Dependencies

### Install Dependencies

In [1]:
!uv pip install -r requirements.txt

Audited 72 packages in 91ms


### Library Imports

In [1]:
from patchright.async_api import async_playwright, Page, Browser
from camoufox.async_api import AsyncCamoufox
import os
import json
import requests
from dotenv import load_dotenv

browser-helper-file.json      OK!
header-network.zip            OK!
headers-order.json            OK!
fingerprint-network.zip       OK!
input-network.zip             OK!


### Environment Variables

In [2]:
AWS_BUCKET_DATA_LOCATION = os.getenv("AWS_BUCKET_DATA_LOCATION")
AWS_BUCKET_METADATA_LOCATION = os.getenv("AWS_BUCKET_METADATA_LOCATION")


## Helper and Functions and Class Definitions

### File Class Object Definition

In [3]:
class File:
    def __init__(
            self,
            order_no : str,
            title : str,
            date : str,
            file_link : str
            ):
        self.order_no = order_no
        self.title = title
        self.date = date
        self.file_link = file_link
    
files : set[File] = set()

### Global Variables

In [4]:
files : set[File] = set()
changes_counter = 0

### JSON Encoder and Progress Loading for File Object

In [5]:
def json_encoder(obj: File):
    """
    Encodes FIle class instance to JSON instance 

    Args:
        obj (File): File object

    Raises:
        TypeError: Occurs when object passed is not an instance of the File class

    Returns:
        dict[str,str]: dictionary for JSON parsing
    """
    if isinstance(obj, File):
        return {
            'Order Number' : obj.order_no,
            'Title' : obj.title,
            'Date Filed' : obj.date,
            'File Link' : obj.file_link
        }
    raise TypeError("Object is not JSON parsable.")

def load_files_from_json(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            for item in data:
                # Reconstruct the File object using the JSON keys
                # We use .get() to avoid errors if a key is missing
                new_file = File(
                    order_no=item.get('Order Number'),
                    title=item.get('Title'),
                    date=item.get('Date Filed'),
                    file_link=item.get('File Link')
                )
                files.add(new_file)
        print(f"Successfully loaded {len(files)} unique bills.")
    except FileNotFoundError:
        print("No existing JSON found. Starting with an empty set.")

### Download File From URL Function

In [ ]:
import os
import cloudscraper

def download(url: str, dest_folder: str):
    """
    Downloads the file from the URL provided using cloudscraper to bypass Cloudflare.

    Inputs:
    url (str): input URL of file
    dest_folder (str): destination folder/directory of downloaded file
    
    Outputs:
    Returns True if the download was successful, and False if not.
    """
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)  
        
    try:
        filename = url.split('/')[-1].replace(" ", "_")  
        file_path = os.path.join(dest_folder, filename)
        
        # Initialize the cloudscraper instance and spoof a standard desktop Chrome browser
        scraper = cloudscraper.create_scraper(
            browser={
                'browser': 'chrome',
                'platform': 'windows',
                'desktop': True
            }
        )
        
        # Fetch the URL using the scraper instead of requests
        r = scraper.get(url, stream=True)
        
        if r.ok:
            with open(file_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024 * 8):
                    if chunk:
                        f.write(chunk)
                        f.flush()
                        os.fsync(f.fileno())
            return True
        else:  
            print(f"Download failed with status code: {r.status_code}")
            return False
            
    except Exception as e:
        print(f"An error occurred: {e}")
        return False

# Example usage:
# success = download("https://ncip.gov.ph/wp-content/uploads/2025/01/AO-01-2024-NCIP-Rules-of-Procedure.pdf", "./downloads")
# print("Success:", success)

## Scraper Stuff

### Actual Scraper

#### File Metadata Reset

In [32]:
# Get updated metadata file from AWS bucket
print(AWS_BUCKET_METADATA_LOCATION)
!rm -rf outputs/*.pdf
# !aws s3 cp {AWS_BUCKET_METADATA_LOCATION}metadata.json outputs  

# Populate files array for cross-checking
load_files_from_json('outputs/metadata.json')
changes_counter = 0


None
Successfully loaded 0 unique bills.


#### Scraper Code

In [33]:

try:
    async with AsyncCamoufox(headless=False, geoip=True) as browser:
        context = await browser.new_context(viewport={"width":1000, "height":500})
        page = await context.new_page()

        await page.goto("https://ncip.gov.ph/administrative-order/")
        await page.wait_for_selector('figure', state='visible', timeout=90000)

        order_list = page.locator('tbody > tr')
        order_count = await order_list.count()
        for i in range(1, order_count):
            order_item = order_list.nth(i)
            order_item_fields = order_item.locator('td')
            order_no = await order_item_fields.nth(0).inner_text()
            title = ""
            file_link = "None"
            try: 
                title = await order_item_fields.nth(1).locator('a').inner_text()
                file_link = await order_item_fields.nth(1).locator('a').get_attribute('href')
            except Exception as e:
                title = await order_item_fields.nth(1).inner_text()
            date = await order_item_fields.nth(2).inner_text()
            if file_link:
                print(file_link)
                download(file_link, 'outputs/administrative_orders/')
            new_file = File(
                order_no=order_no.strip(),
                title=title.strip(),
                date=date.strip(),
                file_link=file_link.strip()
            )
except Exception as e:
    print("Error occurred. Saving progress...")
    print(f"{e}")

# Processing logic (e.g., saving to JSON)
with open('outputs/metadata.json', mode='w', encoding='utf-8') as f:
    json.dump(
        obj=list(files),
        fp=f,
        default=json_encoder,
        indent=4
    )

https://ncip.gov.ph/wp-content/uploads/2025/01/AO-01-2024-NCIP-Rules-of-Procedure.pdf
Download failed with status code: 403
https://ncip.gov.ph/wp-content/uploads/2020/09/ncip_admin_order_01_2020_rules_on_delineation_titling.pdf
Download failed with status code: 403
https://ncip.gov.ph/wp-content/uploads/2025/02/NCIP-AO-1-S.-2021-as-amended-by-CEB-Reso-No.-2023-09-07-063.pdf
Download failed with status code: 403
https://ncip.gov.ph/wp-content/uploads/2025/02/NCIP-Administrative-Order-No.-2-series-of-2019-COC-Guidelines.pdf
Download failed with status code: 403
https://ncip.gov.ph/wp-content/uploads/2020/09/ncip-ao-no-2-s-2018-revised-adsdpp-guidelines.pdf
Download failed with status code: 403
https://ncip.gov.ph/wp-content/uploads/2020/09/ncip-ao-no-1-s-2018-rules-of-procedure.pdf
Download failed with status code: 403
https://ncip.gov.ph/wp-content/uploads/2020/09/ncip-resolution-no.-06-099-2014-eap-amendment.pdf
Download failed with status code: 403
https://ncip.gov.ph/wp-content/uplo

### Upload to AWS Bucket

In [10]:
# Data Sync to S3 Bucket
!aws s3 sync outputs/ {AWS_BUCKET_DATA_LOCATION} --exclude "*" --include "*.pdf"

# Metadata upload to S3
!aws s3 cp outputs/metadata.json {AWS_BUCKET_METADATA_LOCATION}  
print(AWS_BUCKET_METADATA_LOCATION)


upload: outputs/CR00136.pdf to s3://aaia-raw/congress/20th_congress/data/CR00136.pdf
upload: outputs/CR00137.pdf to s3://aaia-raw/congress/20th_congress/data/CR00137.pdf
upload: outputs/HB08102.pdf to s3://aaia-raw/congress/20th_congress/data/HB08102.pdf
upload: outputs/HB08105.pdf to s3://aaia-raw/congress/20th_congress/data/HB08105.pdf
upload: outputs/HB08103.pdf to s3://aaia-raw/congress/20th_congress/data/HB08103.pdf 
upload: outputs/HB08111.pdf to s3://aaia-raw/congress/20th_congress/data/HB08111.pdf 
upload: outputs/HB08106.pdf to s3://aaia-raw/congress/20th_congress/data/HB08106.pdf 
upload: outputs/HB08113.pdf to s3://aaia-raw/congress/20th_congress/data/HB08113.pdf 
upload: outputs/HB08115.pdf to s3://aaia-raw/congress/20th_congress/data/HB08115.pdf 
upload: outputs/HB08114.pdf to s3://aaia-raw/congress/20th_congress/data/HB08114.pdf 
upload: outputs/HB08116.pdf to s3://aaia-raw/congress/20th_congress/data/HB08116.pdf 
upload: outputs/HB08117.pdf to s3://aaia-raw/congress/20th